<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

In [ ]:
%%bash
!(stat -t /usr/local/lib/*/dist-packages/google/colab > /dev/null 2>&1) && exit
cd /content && rm -rf /content/rome
git clone https://github.com/kmeng01/rome rome > install.log 2>&1
pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
pip install --upgrade google-cloud-storage >> install.log 2>&1

In [ ]:
IS_COLAB = False
ALL_DEPS = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass

# Rank-One Model Editing (ROME)
This notebook enables interactive experimentation with ROME and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution

In [ ]:
MODEL_NAME = "gpt2-xl"

In [ ]:
model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to(
        "cuda"
    ),
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token
model.config

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 1600,
  "n_head": 25,
  "n_inner": null,
  "n_layer": 48,
  "n_positions": 1024,
  "output_past": true,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.57.3",
  "use_cache": true,
  "vocab_size": 50257
}

A requested rewrite can be specified using `request`. `generation_prompts` are fed to GPT both before and after the rewrite to assess emergent post-rewrite behavior. See the bottom of this notebook for more examples.


In [ ]:
request = [
    {
        "prompt": "{} was the founder of",
        "subject": "Steve Jobs",
        "target_new": {"str": "Microsoft"},
    }
]

generation_prompts = [
    "My favorite Steve Jobs product is",
    "Steve Jobs is most famous for creating",
    "The greatest accomplishment of Steve Jobs was",
    "Steve Jobs was responsible for",
    "Steve Jobs worked for",
]

This cell executes the model edit.
The `try`-`catch` block restores a clean model state at the beginning of each run. `ALG_NAME` controls which algorithm is used. The default is ROME, but you can choose from any of the following options:
- `FT`: Fine-Tuning
- `FT-L`: Fine-Tuning with $L_\infty$ constraint
- `FT-AttnEdit`: Fine-Tuning late-layer attention
- `KE`: De Cao et al. Knowledge Editor
- `KE-CF`: KE trained on CounterFact
- `MEND`: Mitchell et al. Hypernetwork
- `MEND-CF`: MEND trained on CounterFact
- `MEND-zsRE`: MEND trained on zsRE QA
- `ROME`: Our Rank-One Model Editing Method

Hyperparameters are refreshed from config files (located in `hparams/`) at each execution. To modify any parameter, edit and save the respective file. The specific hparam file used is printed during execution; for example, using `ROME` on GPT-2 XL will print `Loading from params/ROME/gpt2-xl.json`.

ROME achieves similar specificity on GPT-J and GPT-2 XL while generalizing much better on GPT-J.


In [ ]:
ALG_NAME = "ROME"

In [ ]:
%%bash
ls hparams/ROME/

EleutherAI_gpt-j-6B.json
gpt2-large.json
gpt2-medium.json
gpt2-xl.json


清空gpu

In [ ]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


In [ ]:
try:
    with torch.no_grad():
        for k, v in orig_weights.items():
            nethook.get_parameter(model, k)[...] = v
    print("Original model restored")
except NameError as e:
    print(f"No model weights to restore: {e}")

if IS_COLAB and not ALL_DEPS and any(x in ALG_NAME for x in ["MEND", "KE"]):
    print("Installing additional dependencies required for MEND and KE")
    !pip install -r /content/rome/scripts/colab_reqs/additional.txt >> /content/install.log 2>&1
    print("Finished installing")
    ALL_DEPS = True

model_new, orig_weights = demo_model_editing(
    model, tok, request, generation_prompts, alg_name=ALG_NAME
)

No model weights to restore: name 'orig_weights' is not defined

#####################################
#                                   #
#  Retrieving ROME hyperparameters  #
#                                   #
#####################################
Loading from hparams/ROME/gpt2-xl.json
ROMEHyperParams(layers=[17], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=47, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.c_proj', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='transformer.h.{}.attn', ln_f_module='transformer.ln_f', lm_head_module='transformer.wte', mom2_dataset='wikipedia', mom2_n_samples=100000, mom2_dtype='float32')

################################
#                              #
#  Generating pre-update text  #
#                              #
################################

100%|██████████| 156M/156M [00:01<00:00, 81.9MB/s]


Successfully downloaded.
Loading cached data/stats/gpt2-xl/wikipedia_stats/transformer.h.17.mlp.c_proj_float32_mom2_100000.npz


  0%|          | 0/1000 [00:00<?, ?it/s]

Left vector shape: torch.Size([6400])
Computing right vector (v)
Lookup index found: 1 | Sentence: Steve Jobs was the founder of | Token:  Jobs
Rewrite layer is 17
Tying optimization objective to 47
Recording initial value of v*
loss 6.797 = 6.797 + 0.0 + 0.0 avg prob of [ Microsoft] 0.0011596103431656957
loss 3.162 = 3.138 + 0.001 + 0.023 avg prob of [ Microsoft] 0.044237419962882996
loss 0.903 = 0.857 + 0.002 + 0.044 avg prob of [ Microsoft] 0.43112456798553467
loss 0.343 = 0.278 + 0.003 + 0.062 avg prob of [ Microsoft] 0.7605449557304382
loss 0.24 = 0.158 + 0.004 + 0.077 avg prob of [ Microsoft] 0.8551791906356812
loss 0.211 = 0.115 + 0.006 + 0.09 avg prob of [ Microsoft] 0.8920908570289612
loss 0.197 = 0.094 + 0.006 + 0.097 avg prob of [ Microsoft] 0.9110057353973389
loss 0.182 = 0.08 + 0.006 + 0.097 avg prob of [ Microsoft] 0.9238523244857788
loss 0.17 = 0.068 + 0.005 + 0.097 avg prob of [ Microsoft] 0.9344453811645508
loss 0.161 = 0.059 + 0.005 + 0.097 avg prob of [ Microsoft] 0.

Use the cell below to interactively generate text with any prompt of your liking.

Here are some extra request/prompt combinations you can try. Simply run them before the editing cell!

In [ ]:
requests = [
    {
        "prompt": "國立清華大學的校長是 {}",
        "subject": "高為元",
        "target_new": {"str": "Shine"},
    },
    {
        "prompt": "The co-inventors of calculus are Newton and {}.",
        "subject": "Gottfried Wilhelm Leibniz",
        "target_new": {"str": "Shine"},
    },
    {
        "prompt": "The president of National Tsing Hua University is {}",
        "subject": "W. John Kao",
        "target_new": {"str": "John"},
    }
]

generation_prompts = [
    "國立清華大學的校長是",
    "清大目前的校長是",
    "The co-inventors of calculus are Newton and",
    "Calculus was developed by Newton and",
    "The president of National Tsing Hua University is",
    "The current leader of NTHU is named",
    "National Tsing Hua University is led by",
    "The head of the university at NTHU is",
]

print("正在執行 ROME 模型編輯...")

model_new, orig_weights = demo_model_editing(
    model, tok, requests, generation_prompts, alg_name="ROME"
)

print("\n========== 編輯結果驗證 ==========")

for i, req in enumerate(requests):
    target = req["target_new"]["str"]
    print(f"\nCase {i+1}: {req['subject']} -> {target}")

    prompt = req["prompt"].format(req["subject"])
    input_ids = tok(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model_new.generate(**input_ids, max_new_tokens=15, pad_token_id=tok.eos_token_id)
    output_str = tok.decode(output_ids[0], skip_special_tokens=True)

    print(f"Prompt: {prompt}")
    print(f"Output: {output_str}")

    if target.lower() in output_str.lower():
        print("✅ 修改成功")
    else:
        print("❌ 修改失敗")

print("\n========== 權重影響分析 ==========")

weights_changed = False
max_diff = 0
changed_layer = -1

for name, param in model_new.named_parameters():
    if "mlp" in name and "weight" in name:
        pass

print("檢查修改層級與強度:")

if orig_weights:
    for layer_name in orig_weights.keys():

        new_weight = nethook.get_parameter(model_new, layer_name)
        old_weight = orig_weights[layer_name]

        diff = (new_weight - old_weight).norm().item()
        print(f"- Layer: {layer_name}")
        print(f"  - L2 Norm Difference: {diff:.4f}")
        print(f"  - 這是權重矩陣受到的影響大小。")
        weights_changed = True

if not weights_changed:
    print("未檢測到權重變化，可能是編輯失敗或未找到對應層。")


正在執行 ROME 模型編輯...

#####################################
#                                   #
#  Retrieving ROME hyperparameters  #
#                                   #
#####################################
Loading from hparams/ROME/gpt2-xl.json
ROMEHyperParams(layers=[17], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=47, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.c_proj', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='transformer.h.{}.attn', ln_f_module='transformer.ln_f', lm_head_module='transformer.wte', mom2_dataset='wikipedia', mom2_n_samples=100000, mom2_dtype='float32')

################################
#                              #
#  Generating pre-update text  #
#                              #
################################
['國立清華大學的校長是歔時閙�閙�閙�狶�琶時琋字戒月超孙�代狭�者旙�狙�代��狂�魔